In [1]:
# cargo un dataframe en df_epic_ligh.pickle
import pandas as pd
df = pd.read_pickle('df_fe_epic_light.pickle')
df

,product_id,fecha,customer_id,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,...,tn_cat3_vendidas,tn_cat3_vendidas_div,tn_brand_vendidas,tn_brand_vendidas_div,tn_sku_size_vendidas,tn_sku_size_vendidas_div,tn_product_id_vendidas,tn_product_id_vendidas_div,tn_customer_id_vendidas,tn_customer_id_vendidas_div
196482,20012,2017-01,10001,1.0,26,61.053291,61.053291,NaN,HC,ROPA ACONDICIONADOR,...,565.539917,0.107956,476.397278,0.128156,660.797485,0.092393,476.397278,0.128156,250.006836,0.244206
446550,20026,2017-01,10001,0.0,9,11.236500,11.236500,NaN,HC,ROPA LAVADO,...,270.058929,0.041608,420.431793,0.026726,660.797485,0.017004,184.400192,0.060935,250.006836,0.044945
500136,20029,2017-01,10001,0.0,9,11.762890,11.762890,NaN,HC,VAJILLA,...,236.031586,0.049836,420.431793,0.027978,236.031586,0.049836,236.031586,0.049836,250.006836,0.047050
577028,20034,2017-01,10001,0.0,18,65.290680,65.290680,NaN,HC,ROPA LAVADO,...,391.170776,0.166911,391.170776,0.166911,473.071503,0.138014,391.170776,0.166911,250.006836,0.261156
924415,20054,2017-01,10001,0.0,9,4.340390,4.340390,NaN,PC,CABELLO,...,271.988129,0.015958,157.749619,0.027514,157.749619,0.027514,157.749619,0.027514,250.006836,0.017361
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14901562,21159,2019-12,10618,0.0,0,0.000000,0.000000,1.86295,PC,PIEL1,...,0.141940,0.000000,25.496450,0.000000,22.146521,0.000000,0.141940,0.000000,0.000000,NaN
14981796,21170,2019-12,10618,0.0,0,0.000000,0.000000,0.13250,REF,TE,...,0.040780,0.000000,0.062620,0.000000,0.062620,0.000000,0.040780,0.000000,0.000000,NaN
15070540,21182,2019-12,10618,0.0,0,0.000000,0.000000,0.16944,FOODS,SOPAS Y CALDOS,...,0.632830,0.000000,63.202412,0.000000,0.066880,0.000000,0.066880,0.000000,0.000000,NaN
15358702,21222,2019-12,10618,0.0,0,0.000000,0.000000,0.30576,REF,TE,...,0.021840,0.000000,0.062620,0.000000,0.062620,0.000000,0.021840,0.000000,0.000000,NaN


In [2]:
test_index = df.index[df['date_id'] == 33]
train_index = df.index[df['date_id'] <= 31]
train_scaler_index = df.index[df['date_id'] <= 33]

In [3]:
numeric_columns = df.select_dtypes(include=['float64', "float32"]).columns
numeric_columns

Index(['plan_precios_cuidados', 'cust_request_tn', 'tn', 'stock_final',
       'sku_size', 'coseno_fecha', 'seno_fecha', 'cust_request_qty_per_tn',
       'cust_request_tn_minus_tn', 'tn_diff_1', 'tn_diff_2', 'tn_diff_4',
       'tn_diff_11', 'tn_diff_12', 'tn_lag_1', 'tn_lag_2', 'tn_lag_3',
       'tn_lag_4', 'tn_lag_6', 'tn_lag_11', 'tn_lag_12', 'tn_diff_2_lag_1',
       'tn_diff_2_lag_2', 'tn_diff_2_lag_3', 'tn_diff_2_lag_4',
       'tn_diff_2_lag_6', 'tn_diff_2_lag_11', 'tn_diff_2_lag_12',
       'tn_rolling_mean_3', 'tn_rolling_mean_12', 'tn_diff_2_rolling_mean_3',
       'tn_diff_2_rolling_mean_12', 'tn_rolling_std_3', 'tn_rolling_std_12',
       'tn_diff_2_rolling_std_3', 'tn_diff_2_rolling_std_12',
       'tn_rolling_max_3', 'tn_rolling_max_12', 'tn_rolling_max_24',
       'tn_diff_2_rolling_max_3', 'tn_diff_2_rolling_max_12',
       'tn_diff_2_rolling_max_24', 'tn_rolling_min_3', 'tn_rolling_min_12',
       'tn_rolling_min_24', 'tn_diff_2_rolling_min_3',
       'tn_diff_2_roll

In [4]:
# diccionario de transformaciones (key es la columna con la que se entrena y el valor son las columnas que se transforman)
transformations = {
    "tn": [r"tn$", r"cust_request_qty_per_tn$", r"tn_lag_*", r"tn_rolling_mean_*", r"tn_rolling_max_*", r"tn_rolling_min_*", r"tn_.*_vendidas$"],
    "stock_final": [r"stock_final$"],
    "cust_request_tn_minus_tn": [r"cust_request_tn_minus_tn$"],
    "tn_diff_2": [r"tn_diff_*"]
}

In [6]:
import numpy as np
import re

df_scaled = df.copy()
train_scaler_df = df_scaled.loc[train_scaler_index]

prod_stats = (
    train_scaler_df.groupby('product_id')[list(transformations.keys())]
    .agg(['mean', 'std'])
)

def custom_group_stats(group):
    product_id = group.name[1]
    row = {'customer_id': group.name[0], 'product_id': product_id}
    for col in transformations.keys():
        nonzero_count = (group[col] != 0).sum()
        if nonzero_count <= 3:
            mean = prod_stats.loc[product_id, (col, 'mean')]
            std = prod_stats.loc[product_id, (col, 'std')]
        else:
            mean = group[col].mean()
            std = group[col].std()
            if std < 1:
                std = max(group[col].max(), 1)
        row[f"{col}_mean"] = mean
        row[f"{col}_std"] = std
    return pd.Series(row)

group_stats = train_scaler_df.groupby(['customer_id', 'product_id']).apply(custom_group_stats).reset_index(drop=True)

group_stats

/tmp/ipykernel_63171/1928279191.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_stats = train_scaler_df.groupby(['customer_id', 'product_id']).apply(custom_group_stats).reset_index(drop=True)


,customer_id,product_id,tn_mean,tn_std,stock_final_mean,stock_final_std,cust_request_tn_minus_tn_mean,cust_request_tn_minus_tn_std,tn_diff_2_mean,tn_diff_2_std
0,10001.0,20012.0,59.658566,41.559536,62.614479,48.338135,1.709787e+00,3.054462,-1.815812,39.592655
1,10001.0,20026.0,16.077129,11.060883,136.797134,89.375694,1.206827e-02,0.290221,0.859475,13.886817
2,10001.0,20029.0,11.384937,12.246619,47.189888,45.645267,7.611285e-01,1.815592,-1.798741,11.354578
3,10001.0,20034.0,47.266125,33.168709,120.362061,119.764503,5.143320e-01,1.101307,-2.719666,40.195652
4,10001.0,20054.0,6.525206,4.392067,14.110579,28.324512,3.919174e-03,0.127738,0.043754,6.877539
...,...,...,...,...,...,...,...,...,...,...
56025,10637.0,21130.0,0.000725,0.007266,0.833457,0.791133,2.634014e-06,0.000304,-0.000049,0.007604
56026,10637.0,21170.0,0.000489,0.007281,0.687230,0.386788,2.287713e-06,0.000210,-0.000052,0.007104
56027,10637.0,21182.0,0.000475,0.006951,0.338261,0.306644,9.517316e-07,0.000107,-0.000007,0.007711
56028,10637.0,21185.0,0.000509,0.009152,0.359613,0.209884,1.915184e-06,0.000139,-0.000036,0.011341


In [7]:

# Mergear las stats al df original
df_scaled = df_scaled.merge(group_stats, on=['product_id', "customer_id"], how='left')
df_scaled = df_scaled.set_index(df.index)

scaled_cols = {}
for trainer, regex_cols in transformations.items():
    for col in regex_cols:
        # Usar regex para seleccionar las columnas que coinciden
        # chequear si la columna es un regex
        matching_cols = [c for c in numeric_columns if re.match(col, c)]
        if not matching_cols:
            continue  # Si no hay columnas que coincidan, saltar

        # Calcular la media y desviación estándar para cada 
        print(f"Processing trainer: {trainer} with columns: {matching_cols}")
        for match_col in matching_cols:
            mean_col = f"{trainer}_mean"
            std_col = f"{trainer}_std"

        # Escalar las columnas
        for col in matching_cols:
            #scaled_cols[col + "_scaled"] = (df_scaled[col] - df_scaled[trainer + "_mean"]) / df_scaled[trainer + "_std"]
            scaled_cols[col + "_scaled"] = (df_scaled[col]) / df_scaled[trainer + "_std"]
            # reemplazo nan por 0 e inf por 0
            #scaled_cols[col + "_scaled"] = scaled_cols[col + "_scaled"].replace([np.inf, -np.inf], 0)

# Crear un DataFrame con todas las columnas escaladas
scaled_df = pd.DataFrame(scaled_cols, index=df_scaled.index)

# Concatenar de una sola vez
df_scaled = pd.concat([df_scaled, scaled_df], axis=1)
aux_cols = [col + "_mean" for col in list(transformations.keys())] + [col + "_std" for col in list(transformations.keys())]
df_scaled = df_scaled.drop(columns=aux_cols)

df_scaled

Processing trainer: tn with columns: ['tn']
Processing trainer: tn with columns: ['cust_request_qty_per_tn']
Processing trainer: tn with columns: ['tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_4', 'tn_lag_6', 'tn_lag_11', 'tn_lag_12']
Processing trainer: tn with columns: ['tn_rolling_mean_3', 'tn_rolling_mean_12']
Processing trainer: tn with columns: ['tn_rolling_max_3', 'tn_rolling_max_12', 'tn_rolling_max_24']
Processing trainer: tn with columns: ['tn_rolling_min_3', 'tn_rolling_min_12', 'tn_rolling_min_24']
Processing trainer: tn with columns: ['tn_cat1_vendidas', 'tn_cat2_vendidas', 'tn_cat3_vendidas', 'tn_brand_vendidas', 'tn_sku_size_vendidas', 'tn_product_id_vendidas', 'tn_customer_id_vendidas']
Processing trainer: stock_final with columns: ['stock_final']
Processing trainer: cust_request_tn_minus_tn with columns: ['cust_request_tn_minus_tn']
Processing trainer: tn_diff_2 with columns: ['tn_diff_1', 'tn_diff_2', 'tn_diff_4', 'tn_diff_11', 'tn_diff_12', 'tn_diff_2_lag_1', 'tn_diff_

,product_id,fecha,customer_id,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,...,tn_diff_2_rolling_mean_3_scaled,tn_diff_2_rolling_mean_12_scaled,tn_diff_2_rolling_std_3_scaled,tn_diff_2_rolling_std_12_scaled,tn_diff_2_rolling_max_3_scaled,tn_diff_2_rolling_max_12_scaled,tn_diff_2_rolling_max_24_scaled,tn_diff_2_rolling_min_3_scaled,tn_diff_2_rolling_min_12_scaled,tn_diff_2_rolling_min_24_scaled
196482,20012,2017-01,10001,1.0,26,61.053291,61.053291,NaN,HC,ROPA ACONDICIONADOR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
446550,20026,2017-01,10001,0.0,9,11.236500,11.236500,NaN,HC,ROPA LAVADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
500136,20029,2017-01,10001,0.0,9,11.762890,11.762890,NaN,HC,VAJILLA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
577028,20034,2017-01,10001,0.0,18,65.290680,65.290680,NaN,HC,ROPA LAVADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
924415,20054,2017-01,10001,0.0,9,4.340390,4.340390,NaN,PC,CABELLO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14901562,21159,2019-12,10618,0.0,0,0.000000,0.000000,1.86295,PC,PIEL1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14981796,21170,2019-12,10618,0.0,0,0.000000,0.000000,0.13250,REF,TE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15070540,21182,2019-12,10618,0.0,0,0.000000,0.000000,0.16944,FOODS,SOPAS Y CALDOS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15358702,21222,2019-12,10618,0.0,0,0.000000,0.000000,0.30576,REF,TE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# plot en un mismo grafico tn_scaled y tn_rolling_max_12_scaled para product 20001 y customer 10001
import matplotlib.pyplot as plt
def plot_scaled_columns(df, product_id, customer_id, columns):
    subset = df[(df['product_id'] == product_id) & (df['customer_id'] == customer_id)]
    print(subset[columns].iloc[-1])
    plt.figure(figsize=(14, 7))
    for col in columns:
        plt.plot(subset.index, subset[col], label=col)
    plt.title(f'Scaled Columns for Product {product_id} and Customer {customer_id}')
    plt.xlabel('Index')
    plt.ylabel('Scaled Value')
    plt.legend()
    plt.show()
# Llamar a la función para graficar
plot_scaled_columns(df_scaled, product_id=20001, customer_id=10001, columns=['tn', "tn_scaled"])

IndexError: single positional indexer is out-of-bounds

In [9]:
plot_scaled_columns(df_scaled, product_id=20001, customer_id=10002, columns=['tn_scaled'])

IndexError: single positional indexer is out-of-bounds

In [10]:
# el target es tn 2 meses hacia delante
df_scaled['target'] = df_scaled.groupby(['customer_id', 'product_id'])['tn_scaled'].shift(-2)
#df_scaled['target'] = df_scaled.groupby(['customer_id', 'product_id'])['tn_diff_2_scaled'].shift(-2)
df_scaled["target"].describe()



count    1.128649e+06
mean     3.965255e-02
std      2.757812e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      7.462209e+01
Name: target, dtype: float64

In [ ]:
df_scaled["target"]

In [11]:
real_target = pd.DataFrame(df.groupby(['customer_id', 'product_id'])['tn'].shift(-2))

In [12]:
def predict_test(model, X_test, real_target, test_index):
    X_test = df_scaled.loc[test_index].drop(columns=["target", "fecha"])
    predictions = model.predict(X_test)
    df_result = X_test[['customer_id', 'product_id', "tn_scaled", "tn"]].copy()
    df_result['predictions_scaled'] = predictions

    # Mergeá las stats de tn
    df_result = df_result.merge(
        group_stats[['customer_id', 'product_id', 'tn_diff_2_mean', 'tn_diff_2_std', "tn_std", "tn_mean"]],
        #group_stats[['product_id', 'tn_diff_2_mean', 'tn_diff_2_std', "tn_std", "tn_mean"]],
        #group_stats[['customer_id', 'product_id', 'tn_diff_2_median', 'tn_diff_2_iqr', "tn_iqr", "tn_median"]],
        on=['customer_id', 'product_id'],
        #on=['product_id'],
        how='left'
    )
    df_result.set_index(test_index, inplace=True)
    #df_result['predictions'] = df_result['predictions_scaled'] * df_result['tn_std'] + df_result['tn_mean']
    df_result['predictions'] = df_result['predictions_scaled'] * df_result['tn_std']
    #df_result['predictions'] = df_result['predictions_scaled'] * df_result['tn_std'] + df_result['tn_mean']
    # hago la inversa de la diferencia con tn
    #df_result["predictions"] = df_result['predictions'] + df_result['tn']
    # Invertí el escalado

    # multiplico predictions por predictions_df["predictions_binary"] que es el clasificador de 0s
    #df_result["predictions"] = df_result["predictions"] * predictions_df["predictions_binary"]

    df_result = df_result[['customer_id', 'product_id', 'predictions']]
    df_result["target"] = real_target.loc[test_index, "tn"]
    return df_result


# agrupo por product_id y sumo todos los target y predictions
def calculate_total_error(df_result, alpha=1.0):
    grouped = df_result.groupby('product_id').agg({
        'predictions': 'sum',
        'target': 'sum'
    }).reset_index()
    grouped['predictions'] = grouped['predictions'] * alpha
    grouped['abs_error'] = np.abs(grouped['predictions'] - grouped['target'])
    total_error = grouped['abs_error'].sum() / grouped['target'].sum()
    return grouped, total_error

In [13]:
# count targets that are 0s
print("Porcentaje de targets que son 0s:")
print((df_scaled['target'] == 0).mean() * 100)

Porcentaje de targets que son 0s:
74.90314964378597


In [ ]:
# TODO: primero hago un clasificador para predecir si el target es 0 o no, luego hago un regresor para predecir el valor del target
"""
import lightgbm as lgb
X_train = df_scaled.loc[train_index].drop(columns=["target", "fecha"])
y_train = df_scaled.loc[train_index, "target"]
# transformo y_train a 0s y 1s
y_train_binary = (y_train != 0).astype(int)
cat_features = [col for col in X_train.columns if X_train[col].dtype.name == 'category']
train_data = lgb.Dataset(X_train, label=y_train_binary, categorical_feature=cat_features)
y_test = df_scaled.loc[test_index, "target"]
y_test_binary = (y_test != 0).astype(int)
X_test = df_scaled.loc[test_index].drop(columns=["target", "fecha"])
test_data = lgb.Dataset(X_test, label=y_test_binary, categorical_feature=cat_features)
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}
model = lgb.train(params, train_data, valid_sets=[train_data, test_data],
                    valid_names=['train', 'test'], num_boost_round=1000)
"""

In [ ]:
"""
predictions_binary = model.predict(X_test)
predictions_binary = (predictions_binary > 0.5).astype(int)
"""

In [ ]:
"""
predictions_df = pd.DataFrame({
    'customer_id': X_test['customer_id'],
    'product_id': X_test['product_id'],
    'predictions_binary': predictions_binary,
    'target': y_test_binary
})
# matriz de confusion en base a predictions_df
# porcentaje de 0s que son 0s
true_zeros = ((predictions_df['predictions_binary'] == 0) & (predictions_df['target'] == 0)).sum()
# Falsos ceros: predijo 0 pero el target NO es 0
false_zeros = ((predictions_df['predictions_binary'] == 0) & (predictions_df['target'] != 0)).sum()
# Porcentaje de 0s verdaderos que no fueron predichos
no_pred_zeros = ((predictions_df['predictions_binary'] != 0) & (predictions_df['target'] == 0)).sum()

print("cantidad de 0s en prediccions que si son 0s:", true_zeros/ len(predictions_df))
print("cantidad de 0s en prediccions que no son 0s:", false_zeros/ len(predictions_df))
print("cantidad de 0s que no fueron predichos:", no_pred_zeros/ len(predictions_df))
"""

In [14]:

import lightgbm as lgb
y_train = df_scaled.loc[train_index, 'target'].dropna()
X_train = df_scaled.loc[y_train.index].drop(columns=["target", "fecha"])
y_test = df_scaled.loc[test_index, 'target']
X_test = df_scaled.loc[test_index].drop(columns=["target", "fecha"])
cat_features = [col for col in X_train.columns if X_train[col].dtype.name == 'category']
train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
#val_data = lgb.Dataset(y_test.loc[y_test.dropna().index], label=y_test.dropna(), reference=train_data)

X_test = df_scaled.loc[test_index].drop(columns=["target", "fecha"])
# creo callback que se ejecuta cad 200 iteraciones y calcula el total_error
def total_error_callback(env):
    if env.iteration % 200 == 0 and env.iteration > 0:
        df_result = predict_test(env.model, X_test, real_target, test_index)
        grouped, total_error = calculate_total_error(df_result)
        print(f"Iteration {env.iteration}, Total Error: {total_error:.4f}")
        print(grouped.sort_values(by='abs_error', ascending=False).head(10))

# create learning_rate scheduler, arranca en 0.1 y cada iteracion baja 0.99 ** iter
def learning_rate_scheduler(iteration):
    min_lr = 0.001
    new_lr = 0.1 * (0.999 ** iteration)
    new_lr = max(new_lr, min_lr)  # Ensure the learning rate does not go below min_lr
    return new_lr


def num_leaves_scheduler(iteration):
    # Reduce the number of leaves as the iterations increase
    return max(17, 512 - iteration )  # Ensure it doesn't go below 31

callbacks = [
    lgb.log_evaluation(period=50),
    total_error_callback, 
    lgb.reset_parameter(learning_rate=learning_rate_scheduler)
]
model = lgb.train(
    params={
        'objective': 'regression',
        'boosting_type': 'gbdt',
        'metric': 'rmse',
        'num_leaves': 52,
        'learning_rate': 0.05,
        'feature_fraction': 0.2,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        #'max_depth': 10,
        "max_bin": 512,
        "verbose": 0
    },
    train_set=train_data,
    num_boost_round=1000,
    callbacks=callbacks,
    valid_sets=[train_data],
    #early_stopping_rounds=50
)

[50]	training's rmse: 0.204388
[100]	training's rmse: 0.196973
[150]	training's rmse: 0.1919
[200]	training's rmse: 0.187884
Iteration 200, Total Error: 0.4168
    product_id  predictions      target   abs_error
0        20012   373.595393  173.130035  200.465358
6        20077   104.948000   34.588470   70.359529
7        20078   118.157053   51.378662   66.778391
8        20079    73.270965   27.684959   45.586005
2        20029   181.799769  150.648697   31.151073
9        20112    81.657352  107.062630   25.405278
13       20175    37.699189   19.285250   18.413940
18       20242    18.043247    4.036040   14.007206
5        20076    81.546186   68.459221   13.086965
4        20073   109.722206  122.470192   12.747986
[250]	training's rmse: 0.184663
[300]	training's rmse: 0.182041
[350]	training's rmse: 0.179576
[400]	training's rmse: 0.177448
Iteration 400, Total Error: 0.4109
    product_id  predictions      target   abs_error
0        20012   371.621527  173.130035  198.491492
6

In [ ]:
X_test["cat1"].dtype

In [15]:
import numpy as np


df_result = predict_test(model, df_scaled, real_target, test_index)
df_result["abs_error"] = np.abs(df_result['predictions'] - df_result['target'])
#df_result["predictions_binary"] = predictions_df["predictions_binary"]
df_result

,customer_id,product_id,predictions,target,abs_error
196515,10001,20012,55.521308,12.75674,42.764569
446583,10001,20026,13.035222,27.29907,14.263848
500169,10001,20029,11.712554,5.48935,6.223204
924448,10001,20054,6.022670,1.12010,4.902570
1259378,10001,20073,26.120228,23.11125,3.008979
...,...,...,...,...,...
14981750,10606,21170,0.000007,0.00000,0.000007
15070494,10606,21182,0.000018,0.00000,0.000018
15358656,10606,21222,0.000010,0.00000,0.000010
15398844,10606,21228,0.000015,NaN,NaN


In [21]:


grouped, total_error = calculate_total_error(df_result, 0.8)
print("Total Error:", total_error)
print(grouped.sort_values(by='abs_error', ascending=False).head(10))

Total Error: 0.35113792184591963
    product_id  predictions      target   abs_error
0        20012   300.484851  173.130035  127.354816
1        20026   185.308663  235.104187   49.795524
6        20077    82.438432   34.588470   47.849961
7        20078    96.081103   51.378662   44.702441
9        20112    67.710302  107.062630   39.352328
4        20073    87.321697  122.470192   35.148495
8        20079    57.209475   27.684959   29.524516
3        20054   100.848809  121.209099   20.360290
17       20213    38.840388   58.840431   20.000044
10       20126    81.625224   94.952881   13.327657


In [14]:
grouped

,product_id,predictions,target,abs_error
0,20012,361.829951,173.130035,188.699916
1,20026,203.203432,235.104187,31.900755
2,20029,184.695452,150.648697,34.046755
3,20054,114.815239,121.209099,6.393860
4,20073,98.246278,122.470192,24.223914
...,...,...,...,...
64,21170,0.393454,0.040780,0.352674
65,21182,0.488352,0.066880,0.421472
66,21222,0.334612,0.021840,0.312772
67,21228,0.114592,0.000000,0.114592


In [64]:
importance_df = pd.DataFrame({
    'feature': model.feature_name(),
    'importance': model.feature_importance()
}).sort_values(by='importance', ascending=False)
print(importance_df.to_string())

                                feature  importance
80              tn_customer_id_vendidas        1294
84                      tn_lag_1_scaled         902
76                 tn_sku_size_vendidas         859
112  tn_customer_id_vendidas_div_scaled         829
74                    tn_brand_vendidas         782
86                      tn_lag_3_scaled         781
72                     tn_cat3_vendidas         752
70                     tn_cat2_vendidas         725
26                             tn_lag_1         724
127     tn_diff_2_rolling_mean_3_scaled         719
78               tn_product_id_vendidas         694
28                             tn_lag_3         675
110   tn_product_id_vendidas_div_scaled         665
88                      tn_lag_6_scaled         661
1                           customer_id         650
40                    tn_rolling_mean_3         647
82                            tn_scaled         646
36                      tn_diff_2_lag_4         643
94          

In [16]:
# las 100 mejores features son:
top_100_features = importance_df.head(100)
print(top_100_features["feature"].to_list())


['customer_id', 'tn_customer_id_vendidas', 'tn_lag_1_scaled', 'tn_diff_1_scaled', 'tn_diff_2_rolling_mean_3_scaled', 'tn_lag_3_scaled', 'tn_sku_size_vendidas', 'tn_scaled', 'tn_lag_1', 'tn_lag_6_scaled', 'tn_brand_vendidas', 'tn_diff_2_lag_6_scaled', 'tn_brand_vendidas_scaled', 'tn_customer_id_vendidas_div_scaled', 'tn_cat3_vendidas', 'tn_rolling_max_12_scaled', 'tn_diff_2_lag_3_scaled', 'tn_product_id_vendidas_div_scaled', 'tn_product_id_vendidas_scaled', 'tn_brand_vendidas_div_scaled', 'tn_diff_2_rolling_max_3_scaled', 'tn_rolling_min_3', 'tn_diff_2_scaled', 'tn_rolling_max_3_scaled', 'tn_rolling_min_3_scaled', 'brand', 'tn_diff_2_rolling_min_12_scaled', 'tn_diff_2_lag_4_scaled', 'tn_diff_2_lag_4', 'tn_lag_6', 'tn_diff_2_lag_2', 'tn_cat1_vendidas', 'tn_cat1_vendidas_scaled', 'tn_cat2_vendidas_scaled', 'tn_cat3_vendidas_scaled', 'tn_product_id_vendidas', 'tn_rolling_mean_3', 'cat3', 'tn_diff_2_rolling_std_3_scaled', 'tn_cat2_vendidas', 'tn_cat1_vendidas_div_scaled', 'tn_customer_id_ve